# 02 · Métricas lineales y no lineales por ventana
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Valoo-1201/vfc-caos-som/blob/main/notebooks/02_metricas_no_lineales.ipynb)

**Responsable:** Integrante B · **Entrada:** series NN del notebook 01 · **Salida:** `results/features.csv`

| Métrica | Qué mide | Parámetros |
|---|---|---|
| SDNN, RMSSD, pNN50 | Variabilidad lineal (referencia) | — |
| Poincaré SD1, SD2, SD1/SD2 | Variabilidad corta vs. larga | — |
| Entropía Muestral | Irregularidad / imprevisibilidad | m = 2, r = 0.2·SD |
| Entropía Multiescala | Complejidad en varias escalas temporales | escalas 1–10 |
| DFA α1, α2 | Correlaciones fractales | 4–16 y 16–64 latidos |
| Exponente de Lyapunov λ1 | Divergencia de trayectorias vecinas | Rosenstein, m = 5, τ = 1 |

In [ ]:
#@title 0 · Configuración (ejecutar siempre primero)
REPO = "Valoo-1201/vfc-caos-som"   # <- usuario/repositorio en GitHub
RAMA = "main"
USAR_DRIVE = True                  # guarda las series R-R en Google Drive para no descargarlas otra vez
MODO_DEMO = False                  # True = datos sintéticos (solo para probar el código sin internet)

import os, sys, subprocess
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    os.chdir("/content")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")      # Colab -> 🔑 Secretos (solo hace falta si el repo es privado)
    except Exception:
        pass
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    if not os.path.exists("/content/vfc-caos-som"):
        subprocess.run(["git", "clone", "-q", "-b", RAMA, url, "/content/vfc-caos-som"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/vfc-caos-som", "pull", "-q"], check=False)
    os.chdir("/content/vfc-caos-som")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if USAR_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_DIR = "/content/drive/MyDrive/vfc-caos-som-data"
    else:
        DATA_DIR = "/content/data"
else:  # ejecución local: el notebook está en notebooks/
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
    DATA_DIR = "data"

sys.path[:0] = [os.path.abspath("src"), os.getcwd()]
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs("results/figures", exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
SEED = 42
np.random.seed(SEED)
pd.set_option("display.precision", 3)
print("Repositorio:", os.getcwd(), "| Datos:", DATA_DIR, "| Demo:", MODO_DEMO)

In [ ]:
from hrvchaos import data, metrics
from hrvchaos.evaluate import build_feature_table, group_stats
from hrvchaos.plots import plot_mse, plot_feature_boxes

BASES = ["nsr2db", "chf2db", "fantasia"]
RR_DIR = os.path.join(DATA_DIR, "rr")
if MODO_DEMO:
    registros = data.synthetic_cohort(n_per_group=10)
else:
    registros = sum([data.load_database(b, cache_dir=RR_DIR, verbose=False) for b in BASES], [])
print(len(registros), "registros cargados")

## Cálculo de la tabla de características
Una fila por ventana de 2000 latidos (máx. 6 por sujeto). Tarda ~0.5 s por ventana.

In [ ]:
features = build_feature_table(registros, window=2000, max_windows=6, verbose=False)
features.to_csv("results/features.csv", index=False)
print(features.shape, "| tiempo medio por ventana:", round(features.seconds_to_compute.mean(), 2), "s")
features.groupby("group").size()

## Comparación RSN vs. ICC (nivel sujeto)
Mediana de las ventanas de cada sujeto → Mann-Whitney U con corrección de Holm. `cliffs_delta` va de −1 a 1 (|δ| > 0.47 = efecto grande).

In [ ]:
FEATS = metrics.LINEAR + metrics.NONLINEAR
stats = group_stats(features, FEATS, "RSN", "ICC")
stats.to_csv("results/estadistica_RSN_vs_ICC.csv", index=False)
stats

In [ ]:
fig = plot_feature_boxes(features, FEATS, groups=("RSN", "ICC"))
fig.savefig("results/figures/02_boxplots.png", dpi=150); plt.show()
ax = plot_mse(features, groups=[g for g in ["RSN", "ICC", "Joven", "Mayor"] if g in features.group.unique()])
plt.savefig("results/figures/02_mse.png", dpi=150); plt.show()

## Sensibilidad del exponente de Lyapunov
λ1 depende de la dimensión de embebido `m` y del retardo `τ`. Si la diferencia entre grupos cambia de signo con los parámetros, no es una conclusión robusta.

In [ ]:
rng = np.random.default_rng(SEED)
muestra = {g: [r for r in registros if r.group == g] for g in ["RSN", "ICC"]}
filas = []
for g, regs in muestra.items():
    for i in rng.choice(len(regs), size=min(8, len(regs)), replace=False):
        r = regs[i]
        w = r.rr[:2000]
        for m in [4, 5, 6, 7]:
            for tau in [1, 2, 3]:
                filas.append({"group": g, "m": m, "tau": tau, "lyap": metrics.lyapunov_rosenstein(w, m=m, tau=tau)})
sens = pd.DataFrame(filas).pivot_table(index=["m", "tau"], columns="group", values="lyap", aggfunc="median")
sens["diferencia"] = sens["RSN"] - sens["ICC"]
sens

## Prueba con datos sustitutos (IAAFT)
Los sustitutos conservan espectro y distribución pero destruyen la estructura no lineal. Si SampEn o λ1 del dato original caen fuera del rango de sus sustitutos, hay evidencia de dinámica no lineal; si no, la métrica refleja solo propiedades lineales.

In [ ]:
N_SUST = 19   # 19 sustitutos -> prueba bilateral con alfa = 0.10
filas = []
for g, regs in muestra.items():
    for r in regs[:5]:
        w = r.rr[:2000]
        for nombre, f in [("sampen", lambda x: metrics.sample_entropy(x)),
                          ("lyap", lambda x: metrics.lyapunov_rosenstein(x))]:
            orig = f(w)
            sust = np.array([f(metrics.iaaft(w, seed=k)) for k in range(N_SUST)])
            filas.append({"group": g, "subject": r.record, "metrica": nombre, "original": orig,
                          "sust_min": sust.min(), "sust_max": sust.max(),
                          "fuera_de_rango": (orig < sust.min()) or (orig > sust.max())})
sustitutos = pd.DataFrame(filas)
sustitutos.to_csv("results/prueba_sustitutos.csv", index=False)
sustitutos.groupby(["group", "metrica"]).fuera_de_rango.mean().rename("fracción no lineal")